In [ ]:
import torch
import gpytorch

from src.data import make_sine_data, func
from src.quantum_kernel import QuantumKernel
from src.feature_maps import ChebyshevFeatureMap
from src.model import QGP
from src.training import train_model, evaluate_model
from src.plotting import (
    set_plot_style,
    plot_loss,
    plot_regression
)
from src.logger import build_experiment_row, save_experiment

print("Torch:", torch.__version__)
print("GPyTorch imported OK")

In [ ]:
# =========================
# CONFIG HYPERPARAMETERS
# =========================

SEED = 22
torch.manual_seed(SEED)

n_qubits = 2
n_layers = 3
lr = 0.1
epochs = 40
eps = 0.1

phi = torch.arccos

# SANITY CHECK
print("Seed set:", SEED)
print(torch.rand(3))
print("Config loaded:")
print(n_qubits, n_layers, lr, epochs, eps)

In [ ]:
# =========================
# TRAINING DATA
# =========================

X_train, y_train = make_sine_data(
    n_points=30,
    noise=eps
)

# =========================
# TEST DATA
# =========================

X_test = torch.linspace(-1.0, 1.0, 50)
y_test = func(X_test)

# SANITY CHECK
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_train sample:", X_train[:5])
print("y_train sample:", y_train[:5])

import matplotlib.pyplot as plt

plt.scatter(X_train, y_train)
plt.title("Training data sanity check")
plt.show()

In [ ]:
print(X_test[:10])

In [ ]:
# =========================
# MODEL SETUP
# =========================

feature_map = ChebyshevFeatureMap(
    n_qubits=n_qubits,
    n_layers=n_layers,
    phi=phi
)

kernel = QuantumKernel(feature_map)

likelihood = gpytorch.likelihoods.GaussianLikelihood()

model = QGP(X_train, y_train, likelihood, kernel)


# SANITY CHECK
print("Kernel params:", kernel.n_params)
print("Initial kernel norm:", torch.norm(kernel.params).item())

with torch.no_grad():
    K = kernel(X_train, X_train).to_dense()

print("Kernel stats:")
print("min:", K.min().item())
print("max:", K.max().item())
print("mean:", K.mean().item())

import matplotlib.pyplot as plt

plt.imshow(K.detach().numpy())
plt.title("Initial kernel matrix")
plt.colorbar()
plt.show()

In [ ]:
# =========================
# TRAINING
# =========================
import time
start_time = time.time()

losses, initial_params = train_model(
    model=model,
    likelihood=likelihood,
    kernel=kernel,
    X=X_train,
    y=y_train,
    lr=lr,
    epochs=epochs
)

runtime_sec = time.time() - start_time

In [ ]:
# =========================
# KERNEL AFTER TRAINING
# =========================
with torch.no_grad():
    K_trained = kernel(X_train, X_train).to_dense()

print("Trained kernel stats:")
print("min:", K_trained.min().item())
print("max:", K_trained.max().item())
print("mean:", K_trained.mean().item())

plt.imshow(K_trained.detach().numpy())
plt.title("Kernel matrix after training")
plt.colorbar()
plt.show()

In [ ]:
# =========================
# EVALUATION
# =========================

pred_mean, pred_var, mse = evaluate_model(
    model=model,
    likelihood=likelihood,
    X_test=X_test,
    y_test=y_test
)

In [ ]:
print("MSE:", mse)

print("Prediction mean:")
print(pred_mean[:10])

print("Prediction variance:")
print(pred_var[:10])

In [ ]:
# =========================
# PLOTTING
# =========================

set_plot_style()

plot_loss(losses)

plot_regression(
    X_train, y_train,
    X_test, y_test,
    pred_mean, pred_var
)

In [ ]:
# =========================
# LOGGING
# =========================

from src.config import (
    SAVE_MODE,
    SCRATCH_FILE,
    FINAL_FILE
)

row = build_experiment_row(
    n_qubits=n_qubits,
    n_layers=n_layers,
    kernel=kernel,
    lr=lr,
    epochs=epochs,
    N_train=X_train.shape[0],
    N_test=X_test.shape[0],
    noise_eps=eps,
    phi=phi,
    mse=mse,
    runtime_sec=runtime_sec,
    likelihood=likelihood,
    losses=losses,
    initial_params=initial_params,
    seed=SEED
)

print(type(row))
print(row.keys())

save_experiment(
    row=row,
    save_mode=SAVE_MODE,
    scratch_file=SCRATCH_FILE,
    final_file=FINAL_FILE
)